# AI Shopping Assistant

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    The Shopping Assistant                    ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import copy
import json
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_openai import ChatOpenAI

load_dotenv("MY_KEY.env")

YANDEX_CLOUD_FOLDER = os.getenv("YANDEX_CLOUD_FOLDER")
YANDEX_CLOUD_API_KEY = os.getenv("YANDEX_CLOUD_API_KEY")
YANDEX_CLOUD_MODEL = f"gpt://{YANDEX_CLOUD_FOLDER}/gpt-oss-20b/latest"

llm = ChatOpenAI(
    model=YANDEX_CLOUD_MODEL,
    temperature=0,
    api_key=YANDEX_CLOUD_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {
        "id": "p1",
        "name": "Sony WH-1000XM5",
        "category": "headphones",
        "brand": "Sony",
        "price": 349,
        "color": "black",
        "rating": 4.8,
        "tags": ["wireless", "noise-cancelling", "premium"],
    },
    {
        "id": "p2",
        "name": "Sony WH-CH720N",
        "category": "headphones",
        "brand": "Sony",
        "price": 129,
        "color": "blue",
        "rating": 4.4,
        "tags": ["wireless", "budget", "noise-cancelling"],
    },
    {
        "id": "p3",
        "name": "Bose QuietComfort Ultra",
        "category": "headphones",
        "brand": "Bose",
        "price": 379,
        "color": "white",
        "rating": 4.7,
        "tags": ["wireless", "noise-cancelling", "premium"],
    },
    {
        "id": "p4",
        "name": "Apple AirPods Pro 2",
        "category": "earbuds",
        "brand": "Apple",
        "price": 249,
        "color": "white",
        "rating": 4.6,
        "tags": ["wireless", "noise-cancelling", "ios"],
    },
    {
        "id": "p5",
        "name": "Anker Soundcore Liberty 4 NC",
        "category": "earbuds",
        "brand": "Anker",
        "price": 99,
        "color": "black",
        "rating": 4.3,
        "tags": ["wireless", "budget", "noise-cancelling"],
    },
    {
        "id": "p6",
        "name": "Logitech MX Master 3S",
        "category": "mouse",
        "brand": "Logitech",
        "price": 109,
        "color": "graphite",
        "rating": 4.8,
        "tags": ["wireless", "productivity", "premium"],
    },
    {
        "id": "p7",
        "name": "Logitech Pebble 2",
        "category": "mouse",
        "brand": "Logitech",
        "price": 34,
        "color": "white",
        "rating": 4.2,
        "tags": ["wireless", "budget", "portable"],
    },
    {
        "id": "p8",
        "name": "Keychron K2",
        "category": "keyboard",
        "brand": "Keychron",
        "price": 89,
        "color": "black",
        "rating": 4.5,
        "tags": ["wireless", "mechanical", "compact"],
    },
    {
        "id": "p9",
        "name": "NuPhy Air75",
        "category": "keyboard",
        "brand": "NuPhy",
        "price": 139,
        "color": "gray",
        "rating": 4.6,
        "tags": ["wireless", "mechanical", "low-profile"],
    },
    {
        "id": "p10",
        "name": "Amazon Kindle Paperwhite",
        "category": "ereader",
        "brand": "Amazon",
        "price": 149,
        "color": "black",
        "rating": 4.7,
        "tags": ["reading", "portable", "gift"],
    },
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""

    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""

    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""

    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(
        self,
        category: str | None = None,
        query: str = "",
        brand: str | None = None,
        max_price: float | None = None,
        sort_by: str | None = None,
    ) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words):
                continue
            if category and item["category"] != category:
                continue
            if brand and item["brand"].lower() != brand.lower():
                continue
            if max_price is not None and item["price"] > float(max_price):
                continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc":
            results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc":
            results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append(
                {
                    "product_id": product_id,
                    "name": product["name"],
                    "price": product["price"],
                    "quantity": quantity,
                }
            )
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between shopping workflow agents."""

    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)  # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)  # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║             Shopping Assistant Core Workflows                ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Tool-Calling Shopping Flow
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Tool schemas exposed to the LLM.
# The docstrings below describe the tool behavior and parameters.
# convert_to_openai_tool() uses them to build the function-calling schema.


def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """Search for available products according to user wishes (e.g. category, max_price).

    Args:
        query: Free-text search request (e.g. product type, features or name)
        category: Optional category filter of the products (e.g. "headphones", "mouse")
        brand: Optional brand filter (e.g. "Amazon")
        max_price: Optional price limit filter for products
        sort_by: Optional sorting mode. Supported values are price_asc and rating_desc.

    Returns:
        List of matching to user request products.
    """
    ...


def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """Adds a product to the cart by product id.

    Args:
        product_id: Id of product to add (e.g. p6).
        quantity: Amount of products to add.

    Returns:
        A dictionary in which the first boolean flag indicates whether the operation completed successfully (e.g. all products were found),
        and the second value represents the number of products in the cart.
    """
    ...


SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]

RAW_TOOLS_BY_NAME = {"search_products": search_products, "add_to_cart": add_to_cart}


def process_tool_call(
    model_output, tools: ShopTools, state: ShopState, profile_path: Path | None = None
):
    tool_name = (
        model_output["name"] if isinstance(model_output, dict) else model_output.name
    )
    tool_args = dict(
        model_output["args"] if isinstance(model_output, dict) else model_output.args
    )

    if tool_name == "search_products":
        return tools.search_products(**tool_args)

    if tool_name == "add_to_cart":
        return tools.add_to_cart(state, **tool_args)

    if tool_name == "update_profile":
        return update_profile(
            tool_args["key"], tool_args["value"], profile_path or PROFILE_PATH
        )

    raise ValueError(f"Unknown tool: {tool_name}")


def run_shopping_agent(
    user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer
) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """
    messages = [
        SystemMessage(
            content=(
                f"""
            You are a tool-using shop assistant.

            You have access to the following tools:
            {SHOP_TOOLS_SCHEMA}
            """
            )
        ),
        HumanMessage(content=user_message),
    ]

    while True:
        msg = llm_chat(messages, SHOP_TOOLS_SCHEMA)
        messages.append(msg)
        if msg.tool_calls:
            for call in msg.tool_calls:
                tool_result = process_tool_call(call, tools, state)
                if call["name"] == "search_products":
                    state.last_results = tool_result
                tracer.record(call["name"], call["args"], tool_result)

                messages.append(
                    ToolMessage(
                        content=json.dumps(tool_result, ensure_ascii=False),
                        tool_call_id=call["id"],
                    )
                )
        else:
            return msg.content


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Persistent Memory Flow
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATA_DIR = Path("data")
PROFILE_PATH = DATA_DIR / "user_profile.json"
# Common profile fields:
#   name       — user name
#   brand      — preferred brand
#   max_price  — maximum price
#   color      — preferred color
#   category   — category of interest


def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            save_profile({}, path)
            return {}
    return {}


def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(profile, indent=2, ensure_ascii=False), encoding="utf-8")


def update_profile(key: str, value: str, path: Path = PROFILE_PATH) -> dict:
    """Update a field in the user's persistent profile.

    Recommended field names: name, brand, max_price, color, category.

    Args:
        key: Field name to update (e.g. 'brand')
        value: New value for the field (e.g. 'Sony')
    """
    try:
        profile = load_profile(path)
        profile[key] = value
        save_profile(profile, path)
        return {"ok": True, "profile": profile}
    except Exception:
        return {"ok": False, "profile": load_profile(path)}


SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    *SHOP_TOOLS_SCHEMA,
    convert_to_openai_tool(update_profile),
]


def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory-enabled shopping agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    The history stores HumanMessage, AIMessage, and ToolMessage entries
    so the agent can reuse earlier search results in later turns.
    """
    profile = load_profile(profile_path)
    messages = (
        [
            SystemMessage(
                content=(
                    f"""
            You are a tool-using shop assistant with memory.

            You have access to the following tools:
            {SHOP_TOOLS_SCHEMA_WITH_MEMORY}
            
            Tool: update_profile
            Save a user preference or detail to their persistent profile.

            At the start of each conversation, you will be given the user's current profile.
            Use this information to personalize your responses.

            Current user profile: {json.dumps(profile, ensure_ascii=False)}

            RULE: Whenever the user tells you the name of product, brand, max_price, color, category,
            or any personal detail, you MUST immediately call update_profile to save it.
            Call it once per field. Do not ask for confirmation — just save it.

            Common profile fields: name, brand, max_price, color,
            category.
            """
                )
            )
        ]
        + history
        + [HumanMessage(content=user_message)]
    )
    while True:
        msg = llm_chat(messages, SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        messages.append(msg)
        if msg.tool_calls:
            for call in msg.tool_calls:
                tool_result = process_tool_call(call, tools, state, profile_path)
                if call["name"] == "search_products":
                    state.last_results = tool_result

                tracer.record(call["name"], call["args"], tool_result)

                messages.append(
                    ToolMessage(
                        content=json.dumps(tool_result, ensure_ascii=False),
                        tool_call_id=call["id"],
                    )
                )
        else:
            return msg.content, messages[1:]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Multi-Agent Recommendation Flow
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# The recommendation pipeline combines four specialized agents and an orchestrator.
# Its goal is to identify the strongest product match and summarize both advantages and tradeoffs.
# Agents exchange data through a shared AgentContext object.
#
# RetrieverAgent (LLM + tools)
#   Searches for up to 5 relevant products via search_products.
#   Fills ctx.candidates and ctx.max_price.
#   Important: only pass the search tool (not add_to_cart).
#
# ProsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of pros.
#   Fills ctx.pros (dict: product_id -> pros string).
#   Records an "analyze_pros" call in tracer.
#
# ConsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of cons.
#   Fills ctx.cons (dict: product_id -> cons string).
#   Records an "analyze_cons" call in tracer.
#
# RankerAgent (no LLM — logic only)
#   Picks the best product from ctx.candidates:
#     - Filters by ctx.max_price (if set)
#     - Among remaining: highest rating; if tied — lowest price
#   Records a "rank_candidates" call in tracer. Fills ctx.best.
#
# CoordinatorAgent (orchestrator)
#   Runs agents in a chain, maintains a trace list.
#   Trace keys: "delegate_retriever", "delegate_pros", "delegate_cons",
#               "delegate_ranker", "delegate_cart".
#   If the user asks to add the selection to the cart,
#   CoordinatorAgent does it itself via tools.add_to_cart after ranking.
#   Returns AgentResult with response, trace, and context.
#   The response should include: product name, price, rating, pros and cons.


@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


class RetrieverAgent:
    def run(
        self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer
    ) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        RETRIEVER_AGENT_SCHEMA = [convert_to_openai_tool(search_products)]
        messages = [
            SystemMessage(
                content=(
                    f"""
                You are a tool-using Retriever Agent.

                You have access to the following tools:
                {RETRIEVER_AGENT_SCHEMA}
                You should search for up to 5 relevant products via available tools.
                If the user mentions a budget, pass it as max_price.
                """
                )
            ),
            HumanMessage(content=ctx.query),
        ]
        while True:
            msg = llm_chat(messages, RETRIEVER_AGENT_SCHEMA)
            messages.append(msg)
            if msg.tool_calls:
                for call in msg.tool_calls:
                    if call["name"] == "search_products":
                        tool_args = dict(call["args"])
                        tool_result = tools.search_products(**tool_args)
                        ctx.candidates = tool_result[:5]
                        ctx.max_price = tool_args.get("max_price")
                        state.last_results = ctx.candidates

                    tracer.record(call["name"], call["args"], ctx.candidates)
                    messages.append(
                        ToolMessage(
                            content=json.dumps(ctx.candidates, ensure_ascii=False),
                            tool_call_id=call["id"],
                        )
                    )
            else:
                return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        for product in ctx.candidates:
            messages = [
                SystemMessage(
                    content=(
                        """
                    You are Pros Agent.
                    You have no access to any tools.
                    Write 1-2 short sentences describing the advantages of the product.
                    Use only the provided product data.
                    """
                    )
                ),
                HumanMessage(content=json.dumps(product, ensure_ascii=False)),
            ]
            msg = llm_chat(messages)
            ctx.pros[product["id"]] = msg.content.strip()
        tracer.record(
            "analyze_pros",
            {"product_ids": [product["id"] for product in ctx.candidates]},
            ctx.pros,
        )
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""
        for product in ctx.candidates:
            messages = [
                SystemMessage(
                    content=(
                        """
                    You are Cons Agent.
                    You have no access to any tools.
                    Write 1-2 short sentences describing the drawbacks of the product.
                    Use only the provided product data.
                    """
                    )
                ),
                HumanMessage(content=json.dumps(product, ensure_ascii=False)),
            ]
            msg = llm_chat(messages)
            ctx.cons[product["id"]] = msg.content.strip()
        tracer.record(
            "analyze_cons",
            {"product_ids": [product["id"] for product in ctx.candidates]},
            ctx.cons,
        )
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        candidates = ctx.candidates

        if ctx.max_price is not None:
            candidates = [item for item in candidates if item["price"] <= ctx.max_price]

        if not candidates:
            ctx.best = None
            tracer.record("rank_candidates", {"max_price": ctx.max_price}, None)
            return ctx

        ctx.best = sorted(
            candidates, key=lambda item: (-item["rating"], item["price"])
        )[0]

        tracer.record(
            "rank_candidates",
            {
                "max_price": ctx.max_price,
                "candidate_ids": [item["id"] for item in candidates],
            },
            ctx.best,
        )
        return ctx


class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        ctx = AgentContext(query=user_message)
        trace = []

        tracer = ToolTracer()

        trace.append("delegate_retriever")
        ctx = self.retriever.run(ctx, state, tools, tracer)

        trace.append("delegate_pros")
        ctx = self.pros_agent.run(ctx, tracer)

        trace.append("delegate_cons")
        ctx = self.cons_agent.run(ctx, tracer)

        trace.append("delegate_ranker")
        ctx = self.ranker.run(ctx, tracer)

        if (
            ctx.best is not None
            and "add" in user_message.lower()
            and "cart" in user_message.lower()
        ):
            trace.append("delegate_cart")
            ctx.cart_result = tools.add_to_cart(state, ctx.best["id"])

        if ctx.best is None:
            response = "No suitable products found."
        else:
            response = (
                f"Best option: {ctx.best['name']}\n"
                f"Price: ${ctx.best['price']}\n"
                f"Rating: {ctx.best['rating']}\n"
                f"Pros: {ctx.pros.get(ctx.best['id'], 'N/A')}\n"
                f"Cons: {ctx.cons.get(ctx.best['id'], 'N/A')}"
            )

        return AgentResult(response=response, trace=trace, context=ctx)


In [ ]:
# --- Usage examples: tool-calling flow ----------------------------------

# [1.A] Search with price filter
_s1a = ShopState()
_t1a = ToolTracer()
_r1a = run_shopping_agent(
    "Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a
)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState()
_t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b,
    TOOLS,
    _t1b,
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState()
_t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c,
    TOOLS,
    _t1c,
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


=== Tool Call Trace ===
  1. search_products({"category": "headphones", "max_price": 150})
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
OK 1.A
OK 1.B
OK 1.C: 'NuPhy Air75' (rating 4.6)


In [ ]:
# --- Usage examples: memory flow ----------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists():
    _p2a.unlink()
_s2a = ShopState()
_t2a = ToolTracer()
_h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a,
    TOOLS,
    _t2a,
    _h2a,
    _p2a,
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState()
_t2b = ToolTracer()
_h2b = []
_r2b, _ = run_memory_agent(
    "What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b
)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists():
    _p2c.unlink()
_s2c = ShopState()
_h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


OK 2.A
OK 2.B
OK 2.C: added 'Sony WH-CH720N'


In [ ]:
# --- Usage examples: multi-agent flow -----------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(
    query="test",
    candidates=[
        {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
        {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
        {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
    ],
)
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse", "price": 200, "rating": 4.9},
        {"id": "p6", "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7", "name": "Pebble 2", "price": 34, "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")


OK 3.A
OK 3.B
OK 3.C
OK 3.D: context passed correctly, max_price is respected
